# 05 Fundamental rankings

Each endpoint is fetched once. The rankings compare qualified annual statements within industry groups.

Research data only, not investment advice. Each panel refreshes on a schedule; each panel shows its source date.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

These are member research routes (beta). They are listed in /api/v1/catalog member_research_endpoints.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_once, fetch_panel, get_api_key, make_client
client, catalogue = make_client()
api_key = get_api_key()
rank_columns = {
    'category': 'The ranking definition applied to the row.',
    'category_rank': 'The row position within the returned category.',
    'ticker': 'The market identifier for the company.',
    'name': 'The company name supplied by the response.',
    'industry': 'The industry peer group used for comparison.',
    'fiscal_year': 'The fiscal year of the qualified statement.',
    'report_period_end_date': 'The end date of the statement period.',
    'available_at': 'The date or time when the statement became available.',
    'value': 'The ratio calculated from the qualified annual statement.',
    'raw_zscore': 'The ratio distance from the peer mean in sample standard deviations.',
    'score': 'The signed peer score after applying the category direction.',
    'peer_count': 'The number of qualified peers in the comparison.',
}
def ranked_rows(payload, container):
    rows = []
    for category in payload.get(container) or []:
        for rank, row in enumerate(category.get('rows') or [], start=1):
            rows.append({'category': category.get('id'), 'category_rank': rank, **{name: row.get(name) for name in rank_columns if name not in ('category', 'category_rank')}})
    return rows

In [ ]:
picks_frame = fetch_panel('Four industry-peer rankings', lambda: client.member_fundamental_picks(api_key), lambda p: ranked_rows(p, 'categories'), rank_columns, source_date=lambda p: p.get('statement_watermark') or p.get('as_of'), status=lambda p: p.get('state'), message=lambda p: p.get('reason') or (None if any(c.get('rows') for c in p.get('categories') or []) else 'No qualified ranking rows returned.'))

In [ ]:
get_peers = fetch_once(lambda: client.member_fundamental_peers(api_key, 'AAPL'))
peers_frame = fetch_panel('Peers for one company', get_peers, lambda p: ranked_rows(p, 'rankings'), rank_columns, source_date=lambda p: p.get('statement_watermark') or p.get('as_of'), status=lambda p: p.get('state'), message=lambda p: p.get('reason') or p.get('limitation'))

In [ ]:
history_columns = {
    'ticker': 'The market identifier for the peer.',
    'fiscal_year': 'The fiscal year of the statement.',
    'report_period_end_date': 'The end date of the statement period.',
    'available_at': 'The date or time when the statement became available.',
    'roe': 'Net income divided by total equity when meaningful.',
    'asset_turnover': 'Revenue divided by total assets when comparable.',
    'equity_buffer': 'Total equity divided by total assets.',
    'profit_margin': 'Net income divided by revenue when comparable.',
    'payout_ratio': 'Dividends paid divided by net income when meaningful.',
}
peer_history_frame = fetch_panel('Annual history for the peer group', get_peers, lambda p: [{'ticker': peer.get('ticker'), **year} for peer in p.get('peers') or [] for year in peer.get('years') or []], history_columns, source_date=lambda p: p.get('statement_watermark') or p.get('as_of'), status=lambda p: p.get('state'), message=lambda p: p.get('reason') or p.get('limitation'))